# Great Expectations - Валидация музыкальных данных
Задание: Проверяем данные с помощью Great Expectations

In [10]:
import pandas as pd
import great_expectations as gx
import json
from datetime import datetime

In [11]:
df = pd.read_csv('dataset1.csv')
print(f'Размер данных: {df.shape}')

Размер данных: (114000, 21)


In [12]:
# Удаляем лишние колонки
columns_to_drop = [col for col in df.columns if col.startswith('Unnamed') or col == 'index']
if columns_to_drop:
    df = df.drop(columns=columns_to_drop)
print(f'Размер после очистки: {df.shape}')

Размер после очистки: (114000, 20)


In [13]:
# Получаем уникальные жанры
UNIQUE_GENRES = set(df['track_genre'].dropna().unique())
n_genres = len(UNIQUE_GENRES)
print(f'Уникальных жанров: {n_genres}')

Уникальных жанров: 114


In [14]:
# Создаём file context
import shutil
import os

# Очищаем старый context
context_dir = 'great_expectations'
if os.path.exists(context_dir):
    shutil.rmtree(context_dir)

context = gx.get_context(mode='file', context_root_dir=context_dir)
datasource = context.data_sources.add_pandas(name='music_data_source')
asset = datasource.add_dataframe_asset(name='music_data_asset')
batch_request = asset.build_batch_request(options={'dataframe': df})
print('Context создан')

Context создан


In [15]:
# Создаём Expectation Suite
expectation_suite = gx.ExpectationSuite(name='music_data_expectations')
expectation_suite = context.suites.add(expectation_suite)
print(f'Suite создан: {expectation_suite.name}')

Suite создан: music_data_expectations


### Добавление ожиданий в Suite

In [16]:
# Вспомогательные функции для добавления ожиданий
def add_type_and_length(suite, column, type_name, length):
    """Добавляет проверку типа и длины строки."""
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeOfType(column=column, type_=type_name))
    suite.add_expectation(gx.expectations.ExpectColumnValueLengthsToEqual(column=column, value=length))

def add_type_and_length_range(suite, column, type_name, min_len, max_len):
    """Добавляет проверку типа и диапазона длины строки."""
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeOfType(column=column, type_=type_name))
    suite.add_expectation(gx.expectations.ExpectColumnValueLengthsToBeBetween(
        column=column, min_value=min_len, max_value=max_len, strict_min=True, strict_max=True
    ))

def add_type_and_range(suite, column, type_name, min_val, max_val, strict_min=False, strict_max=False):
    """Добавляет проверку типа и диапазона значений."""
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeOfType(column=column, type_=type_name))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
        column=column, min_value=min_val, max_value=max_val, strict_min=strict_min, strict_max=strict_max
    ))

In [17]:
# Проверка обязательных колонок
required_columns = [
    'track_id', 'artists', 'album_name', 'track_name', 'popularity',
    'duration_ms', 'explicit', 'danceability', 'energy', 'key',
    'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'time_signature', 'track_genre'
]

for col in required_columns:
    expectation_suite.add_expectation(
        gx.expectations.ExpectColumnToExist(column=col)
    )
print(f'Добавлено проверок на наличие колонок: {len(required_columns)}')

Добавлено проверок на наличие колонок: 20


In [18]:
# Проверка на отсутствие пропусков
for col in required_columns:
    expectation_suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column=col)
    )
print(f'Добавлено проверок на пропуски: {len(required_columns)}')

Добавлено проверок на пропуски: 20


In [19]:
# Проверка track_id: длина строки строго 22 символа
add_type_and_length(expectation_suite, 'track_id', 'str', 22)

In [20]:
# Проверка artists: длина строки от 2 до 512 символов
add_type_and_length_range(expectation_suite, 'artists', 'str', 2, 512)

In [21]:
# Проверка album_name: длина строки от 2 до 512 символов
add_type_and_length_range(expectation_suite, 'album_name', 'str', 2, 512)

In [22]:
# Проверка track_name: длина строки от 2 до 512 символов
add_type_and_length_range(expectation_suite, 'track_name', 'str', 2, 512)

In [23]:
# Проверка popularity: int от 0 до 100
add_type_and_range(expectation_suite, 'popularity', 'int', 0, 100)

In [24]:
# Проверка duration_ms: int от 0 (не включительно) до 5237760 (включительно)
add_type_and_range(expectation_suite, 'duration_ms', 'int', 0, 5237760, strict_min=True)

In [25]:
# Проверка explicit: bool
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='explicit', type_='bool')
);

In [26]:
# Проверка danceability: float от 0 до 1
add_type_and_range(expectation_suite, 'danceability', 'float', 0, 1)

In [27]:
# Проверка energy: float от 0 до 1
add_type_and_range(expectation_suite, 'energy', 'float', 0, 1)

In [28]:
# Проверка key: int от 0 до 11
add_type_and_range(expectation_suite, 'key', 'int', 0, 11)

In [29]:
# Проверка loudness: float от -45 до 5
add_type_and_range(expectation_suite, 'loudness', 'float', -45, 5)

In [30]:
# Проверка mode: float от 0 до 1
add_type_and_range(expectation_suite, 'mode', 'float', 0, 1)

In [31]:
# Проверка speechiness: float от 0 до 1
add_type_and_range(expectation_suite, 'speechiness', 'float', 0, 1)

In [32]:
# Проверка acousticness: float от 0 до 1
add_type_and_range(expectation_suite, 'acousticness', 'float', 0, 1)

In [33]:
# Проверка instrumentalness: float от 0 до 1
add_type_and_range(expectation_suite, 'instrumentalness', 'float', 0, 1)

In [34]:
# Проверка liveness: float от 0 до 1
add_type_and_range(expectation_suite, 'liveness', 'float', 0, 1)

In [35]:
# Проверка valence: float от 0 до 1
add_type_and_range(expectation_suite, 'valence', 'float', 0, 1)

In [36]:
# Проверка tempo: float от 0 до 256
add_type_and_range(expectation_suite, 'tempo', 'float', 0, 256)

In [37]:
# Проверка time_signature: int от 0 до 5
add_type_and_range(expectation_suite, 'time_signature', 'int', 0, 5)

In [38]:
# Проверка track_genre: str, ограничение на 114 уникальных жанров
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column='track_genre', type_='str')
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(column='track_genre', value_set=list(UNIQUE_GENRES))
)
expectation_suite.add_expectation(
    gx.expectations.ExpectColumnUniqueValueCountToBeBetween(
        column='track_genre', min_value=n_genres, max_value=n_genres, strict_min=False, strict_max=False
    )
)
print(f'Добавлена проверка track_genre с {n_genres} уникальными значениями')

Добавлена проверка track_genre с 114 уникальными значениями


### Сохранение Expectation Suite в JSON

In [39]:
# Сохраняем suite в JSON файл
suite_json = expectation_suite.to_json_dict()
with open('music_data_expectations.json', 'w', encoding='utf-8') as f:
    json.dump(suite_json, f, indent=2, ensure_ascii=False)
print(f'Expectation Suite сохранён в music_data_expectations.json')
print(f'Количество ожиданий: {len(expectation_suite.expectations)}')

Expectation Suite сохранён в music_data_expectations.json
Количество ожиданий: 80


### Запуск проверки и получение результатов

In [40]:
# Запускаем валидацию через validator
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite=expectation_suite
)

validation_result = validator.validate()
print(f'Проверка завершена: {validation_result.success}')

# Сохраняем результат валидации в JSON (для Data Docs используем checkpoint ниже)
results_dict = validation_result.to_json_dict()
with open('music_data_validation_results.json', 'w', encoding='utf-8') as f:
    json.dump(results_dict, f, indent=2, ensure_ascii=False)
print('Результаты проверки сохранены в music_data_validation_results.json')

Calculating Metrics:   0%|          | 0/130 [00:00<?, ?it/s]

Проверка завершена: False
Результаты проверки сохранены в music_data_validation_results.json


### Генерация HTML-отчёта (Data Docs) через Checkpoint

In [41]:
# В GE v0.18+ API значительно изменился
# Пропускаем стандартные Data Docs, используем альтернативный HTML-отчёт ниже
print('Переходим к генерации альтернативного HTML-отчёта...')

Переходим к генерации альтернативного HTML-отчёта...


In [42]:
# Генерация альтернативного HTML-отчёта из результатов валидации
import os

# Читаем результаты валидации
with open('music_data_validation_results.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

# Генерируем HTML-отчёт
passed = sum(1 for r in results.get('results', []) if r.get('success'))
failed = len(results.get('results', [])) - passed
status = '✅ Все проверки пройдены' if results.get('success') else '❌ Обнаружены проблемы'

html = f'''<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Отчёт валидации Great Expectations</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; background: #f5f5f5; }}
        .container {{ max-width: 1200px; margin: 0 auto; background: white; padding: 30px; border-radius: 8px; }}
        h1 {{ color: #333; border-bottom: 3px solid #4CAF50; padding-bottom: 10px; }}
        .summary {{ padding: 20px; border-radius: 8px; margin: 20px 0; }}
        .summary.success {{ background: #e8f5e9; border-left: 4px solid #4CAF50; }}
        .summary.failure {{ background: #ffebee; border-left: 4px solid #f44336; }}
        table {{ width: 100%; border-collapse: collapse; margin: 20px 0; }}
        th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; }}
        th {{ background: #f5f5f5; }}
        .status {{ padding: 4px 12px; border-radius: 4px; font-weight: bold; }}
        .status.pass {{ background: #c8e6c9; color: #2e7d32; }}
        .status.fail {{ background: #ffcdd2; color: #c62828; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 Отчёт валидации данных</h1>
        <div class="summary {'success' if results.get('success') else 'failure'}">
            <h2>Общая сводка</h2>
            <p><strong>Статус:</strong> {status}</p>
            <p><strong>Всего проверок:</strong> {len(results.get('results', []))}</p>
            <p><strong>Пройдено:</strong> {passed}</p>
            <p><strong>Провалено:</strong> {failed}</p>
        </div>
        <h2>Детали проверок</h2>
        <table>
            <thead><tr><th>Статус</th><th>Проверка</th><th>Колонка</th><th>Параметры</th></tr></thead>
            <tbody>'''

for result in results.get('results', []):
    config = result.get('expectation_config', {})
    kwargs = config.get('kwargs', {})
    exp_type = config.get('type', 'unknown')
    success = result.get('success', False)
    column = kwargs.get('column', 'N/A')
    exp_name = exp_type.replace('expect_', '').replace('_', ' ').title()
    params = []
    for k in ['min_value', 'max_value', 'value', 'type_', 'column']:
        if k in kwargs and k != 'column':
            params.append(f"{k}: {kwargs[k]}")
    params_str = ', '.join(params) if params else '-'
    status_class = 'pass' if success else 'fail'
    status_text = '✓' if success else '✗'
    html += f'''<tr><td><span class="status {status_class}">{status_text}</span></td><td>{exp_name}</td><td>{column}</td><td>{params_str}</td></tr>\n'''

html += '''</tbody></table></div></body></html>'''

# Сохраняем отчёт
os.makedirs('great_expectations/uncommitted/data_docs/local_site', exist_ok=True)
with open('great_expectations/uncommitted/data_docs/local_site/index.html', 'w', encoding='utf-8') as f:
    f.write(html)

print(f'HTML-отчёт сгенерирован')
print(f'Путь: {os.path.abspath("great_expectations/uncommitted/data_docs/local_site/index.html")}')
print(f'Размер: {os.path.getsize("great_expectations/uncommitted/data_docs/local_site/index.html")} байт')

HTML-отчёт сгенерирован
Путь: C:\Projects\yp-sprint-5-practice-1\great_expectations\uncommitted\data_docs\local_site\index.html
Размер: 9946 байт
